# Transcribe and evaluate Dutch audio with Whisper on vLLM

First follow [README setup and audio files](../README.md). Start `./scripts/start-mlx-vllm.sh openai/whisper-large-v3` on Apple Silicon or `./scripts/start-cuda-vllm.sh openai/whisper-large-v3` on Linux CUDA, after stopping any Gemma server on the same port.

Select **DIY-AI workshop (Python 3.12)** and run all cells in order. Set `VLLM_BASE_URL`, `WHISPER_MODEL`, and optionally `VLLM_API_KEY` before starting Jupyter, or edit the setup cell. Project paths work from either the repository root or `notebooks/`.

The notebook uploads the complete MP3 to `/v1/audio/transcriptions` with Dutch as the language. The final cells compare the result with both curated references using word error rate (WER); the curated TXT is primary.

API reference: [vLLM Speech to Text](https://docs.vllm.ai/en/stable/serving/online_serving/speech_to_text/).


**Saved outputs are historical workshop runs, retained as evidence.** They are not a test of your current machine or the selected kernel. Restart the kernel and run all cells to collect your own results. The presentation keeps its dated measurements.

The retained Whisper output was generated on Apple Silicon with vLLM Metal.


In [9]:
import os
import sys
from pathlib import Path

import requests

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_URL = (
    os.environ.get("VLLM_BASE_URL", "http://localhost:8000")
    .rstrip("/")
    .removesuffix("/v1")
)
MODEL_NAME = os.environ.get("WHISPER_MODEL", "openai/whisper-large-v3")
API_KEY = os.environ.get("VLLM_API_KEY", "")
headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
DATA_DIR = PROJECT_ROOT / "data/UMC Utrecht - Jeroen Hendrikse - radiologieverslag"
AUDIO_PATH = DATA_DIR / f"{DATA_DIR.name}.mp3"
CURATED_TXT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.txt"
CURATED_SRT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.srt"
LANGUAGE = "nl"

for path in (AUDIO_PATH, CURATED_TXT_PATH, CURATED_SRT_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}. "
            "Download the workshop audio and curated TXT/SRT files and place them "
            "as shown in README.md, section 6 (Audio files)."
        )

In [10]:
response = requests.get(f"{BASE_URL}/v1/models", headers=headers, timeout=10)
response.raise_for_status()
models = [item["id"] for item in response.json()["data"]]
print("Available models:", models)
if MODEL_NAME not in models:
    raise RuntimeError(
        f"{MODEL_NAME} is not served at {BASE_URL}. "
        "Start a Whisper server or set WHISPER_MODEL to its advertised ID."
    )
print(f"Model: {MODEL_NAME}")

Model: openai/whisper-large-v3


In [11]:
parameters = {"model": MODEL_NAME, "response_format": "json", "temperature": 0}
if LANGUAGE is not None:
    parameters["language"] = LANGUAGE

with AUDIO_PATH.open("rb") as audio_file:
    response = requests.post(
        f"{BASE_URL}/v1/audio/transcriptions",
        headers=headers,
        files={"file": (AUDIO_PATH.name, audio_file, "audio/mpeg")},
        data=parameters,
        timeout=(10, 600),
    )
response.raise_for_status()
transcription = response.json()["text"].strip()
if not transcription:
    raise RuntimeError("The server returned an empty transcription.")

In [12]:
print(transcription)

Ja, dat zou moeten werken. Ja. Oké, dus wat er eigenlijk... Ik kijk in de verslagleiste en ik zie een MRI hersenen staan van een bepaalde aanvrager. Ik weet dat het een hersentumor follow-up waarschijnlijk zal zijn. Dus ik klik die patiënt aan en zie dan inderdaad een follow-up staan van een patiënt met drie maanden geleden ook een scan. Dus ik heb even... Heel snel zie ik het scanlijstje met verschillende data gegeven.  Van eerdere onderzoeken. Daarin zie je bijvoorbeeld dat er een jaar lang. Al follow-up wordt gemaakt. Dus een jaar geleden was blijkbaar de ontdekking. Van de tumor met een aantal onderzoeken. Extern. Ik kijk daar later naar. Ik heb het in mijn geheugen al opgeslagen. Als ik de uitgangssituatie wil weten. Moet ik de scan van een jaar terug ernaast hangen. Nu is eigenlijk alleen relevant. Follow-up. Verandert er iets ten opzichte van drie maanden terug. Dat is eigenlijk de uitgangsdata. De grove data van momenten.  dat ik ga kijken, dan hang ik ze voor. Ga ik in princip

## Compare with curated transcriptions

WER = (substitutions + deletions + insertions) / reference word count.
We use minimum word-level edit distance after Unicode normalization, case folding, and removal of punctuation; hyphens split words.
Accents, numbers, repeated words, and fillers are retained. SRT cue numbers and timestamps are excluded.
This measures transcription differences, including any editorial changes in the curated references.

Saved results were re-evaluated on **23 October 2026** using the updated curated TXT and SRT files. The original model transcription and inference timing are retained; inference was not rerun. The final cell includes the transcription, reference SHA-256 hashes, reference agreement, and both WER scores.


In [ ]:
import re
import unicodedata


def srt_text(contents):
    """Extract subtitle text, excluding each cue's index and timing line."""
    cues = re.split(r"\n\s*\n", contents.strip())
    text = []
    for cue in cues:
        lines = cue.splitlines()
        timing = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if timing is None:
            raise ValueError(f"Subtitle cue has no timestamp: {cue!r}")
        text.extend(lines[timing + 1 :])
    return " ".join(text)


def normalize_words(text):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFKC", text).casefold())


def word_errors(reference, hypothesis):
    """Levenshtein alignment; ties prefer fewer substitutions, then deletions."""
    # Each tuple stores (total errors, substitutions, deletions, insertions).
    previous = [(i, 0, 0, i) for i in range(len(hypothesis) + 1)]
    for i, reference_word in enumerate(reference, start=1):
        current = [(i, 0, i, 0)]
        for j, hypothesis_word in enumerate(hypothesis, start=1):
            if reference_word == hypothesis_word:
                current.append(previous[j - 1])
                continue
            cost, substitutions, deletions, insertions = previous[j - 1]
            substitute = (cost + 1, substitutions + 1, deletions, insertions)
            cost, substitutions, deletions, insertions = previous[j]
            delete = (cost + 1, substitutions, deletions + 1, insertions)
            cost, substitutions, deletions, insertions = current[j - 1]
            insert = (cost + 1, substitutions, deletions, insertions + 1)
            current.append(min(substitute, delete, insert))
        previous = current
    errors, substitutions, deletions, insertions = previous[-1]
    if not reference:
        raise ValueError("The reference contains no words; WER is undefined.")
    return {
        "reference_words": len(reference),
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "errors": errors,
        "wer": errors / len(reference),
    }


references = {
    "Curated TXT (primary)": CURATED_TXT_PATH.read_text(encoding="utf-8-sig"),
    "Curated SRT": srt_text(CURATED_SRT_PATH.read_text(encoding="utf-8-sig")),
}

In [ ]:
import hashlib

print("Transcription:")
print(transcription)
print("\nReference SHA-256:")
for label, path in (("TXT", CURATED_TXT_PATH), ("SRT", CURATED_SRT_PATH)):
    print(f"  {label}: {hashlib.sha256(path.read_bytes()).hexdigest()}")
print()

hypothesis_words = normalize_words(transcription)
reference_words = {name: normalize_words(text) for name, text in references.items()}
comparison_results = {
    name: word_errors(words, hypothesis_words)
    for name, words in reference_words.items()
}

print(f"Whisper transcription: {len(hypothesis_words)} normalized words")
print(
    f"Curated TXT and SRT match after normalization: {len({tuple(words) for words in reference_words.values()}) == 1}"
)
for name, result in comparison_results.items():
    print(f"\n{name} — Word error rate (WER): {result['wer']:.2%}")
    print(
        f"  ({result['substitutions']} substitutions + "
        f"{result['deletions']} deletions + {result['insertions']} insertions)"
        f" / {result['reference_words']} reference words"
        f" = {result['errors']} / {result['reference_words']}"
    )

Transcription:
Ja, dat zou moeten werken. Ja. Oké, dus wat er eigenlijk... Ik kijk in de verslagleiste en ik zie een MRI hersenen staan van een bepaalde aanvrager. Ik weet dat het een hersentumor follow-up waarschijnlijk zal zijn. Dus ik klik die patiënt aan en zie dan inderdaad een follow-up staan van een patiënt met drie maanden geleden ook een scan. Dus ik heb even... Heel snel zie ik het scanlijstje met verschillende data gegeven.  Van eerdere onderzoeken. Daarin zie je bijvoorbeeld dat er een jaar lang. Al follow-up wordt gemaakt. Dus een jaar geleden was blijkbaar de ontdekking. Van de tumor met een aantal onderzoeken. Extern. Ik kijk daar later naar. Ik heb het in mijn geheugen al opgeslagen. Als ik de uitgangssituatie wil weten. Moet ik de scan van een jaar terug ernaast hangen. Nu is eigenlijk alleen relevant. Follow-up. Verandert er iets ten opzichte van drie maanden terug. Dat is eigenlijk de uitgangsdata. De grove data van momenten.  dat ik ga kijken, dan hang ik ze voor. G